In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U google-generativeai requests

In [ ]:
# Cell 2 — API key setup (3-strategy fallback — no getpass)
import os

GEMINI_API_KEY = None

# Strategy 1: Google Colab Secrets (add key named GEMINI_API_KEY via the key icon sidebar)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if GEMINI_API_KEY:
        print("API key loaded from Colab Secrets.")
except Exception:
    pass  # Not in Colab, or secret not set

# Strategy 2: Environment variable (set GEMINI_API_KEY before launching Jupyter)
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    if GEMINI_API_KEY:
        print("API key loaded from environment variable.")

# Strategy 3: Direct assignment — paste your key between the quotes and re-run this cell
if not GEMINI_API_KEY:
    GEMINI_API_KEY = ""  # <-- paste your key here
    if GEMINI_API_KEY:
        print("API key loaded from direct assignment.")

if not GEMINI_API_KEY:
    raise ValueError(
        "No API key found. Use one of:\n"
        "  1. Colab: click the key icon in the sidebar, add secret named GEMINI_API_KEY\n"
        "  2. Local: set environment variable GEMINI_API_KEY before launching Jupyter\n"
        "  3. Edit this cell: paste your key into GEMINI_API_KEY = '' above"
    )

In [ ]:
# Cell 3 — Configure Gemini SDK
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
print("Gemini configured successfully.")

In [ ]:
# Cell 4 — Embedder helper + tool schema
import requests

EMBEDDER_URL = "https://feel-dork-entree.ngrok-free.dev/ask"  # update if ngrok restarts

def ask_embedder(query: str) -> str:
    try:
        response = requests.post(EMBEDDER_URL, json={"question": query}, timeout=10)
        response.raise_for_status()
        return response.json().get("answer", "No answer field in response.")
    except requests.exceptions.Timeout:
        return "Error: The embedder API timed out."
    except requests.exceptions.RequestException as e:
        return f"Error calling embedder API: {e}"

pediatric_tool = genai.protos.Tool(
    function_declarations=[
        genai.protos.FunctionDeclaration(
            name="pediatric_fact_lookup",
            description=(
                "Look up a factual answer from the pediatric curriculum knowledge base. "
                "Use this whenever the user asks a question about pediatric medicine, "
                "child health, developmental milestones, vital signs, dosing, or any "
                "topic covered in the pediatric curriculum."
            ),
            parameters=genai.protos.Schema(
                type=genai.protos.Type.OBJECT,
                properties={
                    "query": genai.protos.Schema(
                        type=genai.protos.Type.STRING,
                        description="The pediatric question to look up."
                    )
                },
                required=["query"]
            )
        )
    ]
)

print("Tool and helper function defined.")

In [ ]:
# Cell 5 — Create Gemini model with tool
model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    tools=[pediatric_tool],
    system_instruction=(
        "You are a helpful pediatric medical education assistant. "
        "You have access to a validated pediatric curriculum knowledge base via the "
        "pediatric_fact_lookup tool. Always use this tool when the user asks a factual "
        "question about pediatrics. Synthesise the retrieved fact into a clear, "
        "conversational answer. If the knowledge base returns no relevant answer, "
        "say so honestly and suggest the user consult a clinical resource."
    )
)

print(f"Model ready: {model.model_name}")

In [ ]:
# Cell 6 — Chat turn handler (manages Gemini function-calling protocol)
from google.generativeai.types import Part

def chat_turn(chat, user_message: str) -> str:
    response = chat.send_message(user_message)

    while True:
        fn_calls = [
            p for p in response.parts
            if hasattr(p, "function_call") and p.function_call.name
        ]

        if not fn_calls:
            text_parts = [p.text for p in response.parts if hasattr(p, "text") and p.text]
            return "\n".join(text_parts) if text_parts else "(No response)"

        fn_responses = []
        for part in fn_calls:
            name = part.function_call.name
            args = dict(part.function_call.args)

            if name == "pediatric_fact_lookup":
                result = ask_embedder(args.get("query", ""))
            else:
                result = f"Unknown tool: {name}"

            fn_responses.append(
                Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(fn_responses)

print("Chat handler ready.")

In [ ]:
# Cell 7 — Conversation loop (run this cell manually; type 'quit' to stop)
chat = model.start_chat()

print("Pediatric Curriculum Chatbot (gemini-2.0-flash)")
print("Type 'quit' or 'exit' to stop.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        print("\n[Input stream ended — stopping chatbot]")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    print("Assistant: ", end="", flush=True)
    try:
        reply = chat_turn(chat, user_input)
        print(reply)
    except Exception as e:
        print(f"[Error: {e}]")

    print()